# 01 — Exploratory Data Analysis

This notebook loads the synthetic data and generates basic statistics and visualisations for all four node types. Run this before any modelling to understand distributions.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src.graph.builder import load_data

workers, platforms, skills, districts, edges = load_data(use_synthetic=True)
print('Workers:', len(workers))
print('Platforms:', len(platforms))
print('Skills:', len(skills))
print('Districts:', len(districts))
print('Edges:', len(edges))

In [ ]:
workers.describe()

In [ ]:
# Distribution of income volatility across worker clusters
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].barh(workers['label'], workers['income_volatility'], color='steelblue')
axes[0].set_xlabel('Income volatility')
axes[0].set_title('Income volatility by cluster')

axes[1].barh(workers['label'], workers['avg_monthly_income'], color='darkorange')
axes[1].set_xlabel('Avg monthly income (Rs)')
axes[1].set_title('Average income by cluster')

axes[2].barh(workers['label'], workers['estimated_count'], color='seagreen')
axes[2].set_xlabel('Estimated count')
axes[2].set_title('Population by cluster')

plt.tight_layout()
plt.savefig('../logs/eda_workers.png', bbox_inches='tight')
plt.show()

In [ ]:
# Platform exit risk vs digital integration
fig, ax = plt.subplots(figsize=(7, 4))
for _, row in platforms.iterrows():
    color = 'red' if row['exit_probability'] >= 0.6 else 'steelblue'
    ax.scatter(row['digital_integration_score'], row['exit_probability'], s=row['worker_count']/500,
               color=color, alpha=0.7, label=row['name'])
    ax.annotate(row['name'], (row['digital_integration_score'], row['exit_probability']),
                textcoords='offset points', xytext=(6, 0), fontsize=8)

ax.axhline(0.6, color='red', linestyle='--', linewidth=0.8, label='High exit threshold')
ax.set_xlabel('Digital integration score')
ax.set_ylabel('Exit probability')
ax.set_title('Platform risk landscape (bubble size = worker count)')
ax.set_xlim(0, 1.1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../logs/eda_platforms.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation between features and risk label
feature_cols = ['income_volatility', 'platform_dependency', 'avg_monthly_income', 'estimated_count']
corr = workers[feature_cols + ['risk_label']].corr()['risk_label'].drop('risk_label')
print('Correlation with risk_label:')
print(corr.sort_values(ascending=False).to_string())

In [ ]:
# Edge type distribution
print(edges['edge_type'].value_counts().to_string())